# logsumexp-cross-entropy composite — cx15: max-shift then exp/sum/log — the LSE primitive

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `broadcasting-rules`, `logsumexp-cross-entropy`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "logsumexp-cross-entropy"
DD_ATOM_IDS = ["broadcasting-rules", "logsumexp-cross-entropy"]
DD_SUBTOPICS = ["Numpy: Vectorization and broadcasting", "Loss: logsumexp cross-entropy"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## The LSE max-shift = subtract row max, then exp/sum/log + add back

1. **`broadcasting-rules`** — `(B, C) - (B, 1)` is the canonical
   keepdim-shaped broadcast that lets the per-row max subtract from every
   class logit in that row.
2. **`logsumexp-cross-entropy`** — the numerically stable form of
   `log(sum(exp(x)))` is `log(sum(exp(x - max(x)))) + max(x)`. The max-shift
   keeps every exp argument ≤ 0, avoiding overflow.

Composition: the broadcast-subtract is what *makes* the max-shift work
across a batch — without `keepdim=True` on the max, you can't subtract it
row-wise. This drill isolates the max-shift core of LSE.


### Composite Exercise — max-shift then exp/sum/log — the LSE primitive

**Atoms exercised together**: `broadcasting-rules`, `logsumexp-cross-entropy`

Implement `cx15_lse_per_row(logits)` for `logits` shape `(B, C)`.

Return a 1-D tensor of shape `(B,)` where entry `i` is the numerically
stable `logsumexp(logits[i])`. **Build it by hand** as:

1. Compute `m = logits.max(dim=-1, keepdim=True).values` → shape `(B, 1)`.
2. Shifted: `shifted = logits - m` → shape `(B, C)` (broadcast subtract).
3. `lse = (shifted.exp().sum(dim=-1)).log() + m.squeeze(-1)` → shape `(B,)`.

Constraints:
- Do NOT call `torch.logsumexp`. You're writing it.
- Must handle logits up to ~10000 without overflow (the test checks).
- Output shape must be `(B,)` (squeezed). The kept-dim is only used for the
  subtract; the final add restores then squeezes.


In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx15_lse_per_row(logits: Tensor) -> Tensor:
    """Per-row logsumexp built from max-shift + exp + sum + log."""
    raise NotImplementedError()


def _test_cx15():
    import inspect
    import math
    # --- forbid torch.logsumexp (must write it by hand) ---
    src = inspect.getsource(cx15_lse_per_row)
    assert 't.logsumexp' not in src and 'torch.logsumexp' not in src, (
        'must write LSE by hand, not call torch.logsumexp'
    )

    # --- uniform logits: LSE per row = log(C) ---
    logits = t.zeros(4, 3)
    out = cx15_lse_per_row(logits)
    assert out.shape == (4,), f'shape {out.shape}'
    assert t.allclose(out, t.full((4,), math.log(3)), atol=1e-5), f'uniform: {out}'

    # --- compare to torch.logsumexp witness ---
    rng = t.Generator().manual_seed(0)
    L = t.randn(6, 8, generator=rng) * 5.0
    ours = cx15_lse_per_row(L)
    ref = t.logsumexp(L, dim=-1)
    assert ours.shape == ref.shape
    assert t.allclose(ours, ref, atol=1e-5), f'mismatch vs torch.logsumexp: {ours} vs {ref}'

    # --- THE point: must survive huge logits (overflow protection) ---
    big = t.tensor([[10000.0, 9999.0, 10001.0, 9998.0],
                    [-10000.0, -9999.0, -10001.0, -9998.0]])
    out_big = cx15_lse_per_row(big)
    assert t.isfinite(out_big).all(), f'overflow! got {out_big}'
    ref_big = t.logsumexp(big, dim=-1)
    assert t.allclose(out_big, ref_big, atol=1e-3), f'{out_big} vs {ref_big}'

    # --- shape contract: (B, C) → (B,) ---
    out2 = cx15_lse_per_row(t.randn(11, 4, generator=rng))
    assert out2.shape == (11,)

    # --- atom-coverage: broadcasting-rules must do real work (keepdim=True over C, then subtract) ---
    # Forbid manual expansion that would sidestep broadcasting.
    _src2 = inspect.getsource(cx15_lse_per_row)
    assert '.expand(' not in _src2 and 'expand_as' not in _src2, (
        'must rely on broadcasting-rules, not torch.expand'
    )
    assert 'broadcast_to' not in _src2, 'must rely on broadcasting-rules, not broadcast_to'
    assert 'repeat(' not in _src2, (
        'must use broadcasting (keepdim=True + subtract), not einops.repeat to materialize the max'
    )
    # Shape misalignment: (B, C) with B=1 must still work (broadcasting (1,1) over (1,C)).
    _l1 = t.tensor([[1.0, 2.0, 3.0, 4.0, 5.0]])
    _o1 = cx15_lse_per_row(_l1)
    assert _o1.shape == (1,), f'(1, C) input must yield (1,): got {_o1.shape}'
    assert t.allclose(_o1, t.logsumexp(_l1, dim=-1), atol=1e-5)
    # Asymmetric shape (B != C) — guards against any axis confusion.
    _l2 = t.randn(3, 7)
    assert t.allclose(cx15_lse_per_row(_l2), t.logsumexp(_l2, dim=-1), atol=1e-5)
    _l3 = t.randn(7, 3)
    assert t.allclose(cx15_lse_per_row(_l3), t.logsumexp(_l3, dim=-1), atol=1e-5)

    _dd_passed.add('cx15')

_test_cx15()

<details><summary>Show solution — cx15</summary>

```python
def cx15_lse_per_row(logits: Tensor) -> Tensor:
    # atom: broadcasting-rules — max with keepdim gives (B, 1), then
    # (B, C) - (B, 1) right-aligns and broadcasts across the C axis.
    m = logits.max(dim=-1, keepdim=True).values
    shifted = logits - m
    # atom: logsumexp-cross-entropy — exp/sum/log + add back the shift.
    return (shifted.exp().sum(dim=-1)).log() + m.squeeze(-1)

```

The `keepdim=True` on the max is what makes the subtract broadcast across
the class axis — that's the broadcasting-rules half. The max-shift + exp +
sum + log + add-back is the logsumexp-cross-entropy half. Without the shift,
`exp(10000)` overflows in float32; with it, every exp arg is ≤ 0.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx15'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx15',
        'subtopics': ["Numpy: Vectorization and broadcasting", "Loss: logsumexp cross-entropy"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()